# Notebook 07 – Dự đoán cho tập test của cuộc thi (nộp bài Kaggle)

Đây là **kiểm định độc lập bên ngoài**: khi nộp bài, Kaggle chạy lại toàn bộ notebook trên **tập test ẩn có nhãn thật** (người nộp không nhìn thấy nhãn) và chấm bằng chỉ số ổn định Gini theo tuần.

Quy trình áp dụng **đúng** các bước như với dữ liệu train: cùng hàm gộp bảng, cùng 3 tỷ lệ, cùng hàm gộp `credit_bureau_a_1`, cùng danh sách đặc trưng và **mô hình cuối** (đặc trưng gốc + credit bureau, notebook 05). Không có đặc trưng hay nhãn nào được tạo thêm.
Kết quả: `submission.csv` gồm `case_id`, `score` (xác suất vỡ nợ).

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import numpy as np, pandas as pd, polars as pl, joblib
import utils

OUT_DIR = utils.get_out_dir()
USE_FINAL = (OUT_DIR / "final_model.joblib").exists()
pack = joblib.load(OUT_DIR / ("final_model.joblib" if USE_FINAL else "best_model.joblib"))
model, feats = pack["model"], pack["features"]
print("Mo hinh:", pack["name"], "| so dac trung:", len(feats))

utils.SPLIT = "test"                                   # doc test_*.parquet
try:
    raw = utils.build_features(["static_0", "static_cb_0"], ["applprev_1", "person_1"], None)
    if USE_FINAL:
        cbf, _ = utils.credit_bureau_features(utils.load_base(), utils.load_definitions())
        raw = raw.join(cbf, on="case_id", how="left")
finally:
    utils.SPLIT = "train"
print("Tap test:", raw.shape)

def add_ratio(df, num, den, name):                     # giong het notebook 02
    if num in df.columns and den in df.columns:
        return df.with_columns((pl.col(num) / pl.when(pl.col(den) > 0).then(pl.col(den))).alias(name))
    return df
raw = add_ratio(raw, "annuity_780A", "maininc_215A", "ratio_annuity_income")
raw = add_ratio(raw, "credamount_770A", "maininc_215A", "ratio_credit_income")
raw = add_ratio(raw, "annuity_780A", "credamount_770A", "ratio_annuity_credit")

In [ ]:
# Dua ve dung cot va kieu du lieu nhu luc huan luyen
schema = pl.read_parquet_schema(OUT_DIR / ("features_v2.parquet" if USE_FINAL else "features_v1.parquet"))
print("Cot thieu trong tap test (dien rong):", len([c for c in feats if c not in raw.columns]))
parts, CHUNK = [], 100_000                              # du doan theo khoi de tiet kiem bo nho
for s0 in range(0, raw.height, CHUNK):
    X = utils.align_features(raw.slice(s0, CHUNK).to_pandas(), feats, schema)
    parts.append(pd.DataFrame({"case_id": X["case_id"].astype("int64"), "score": model.predict_proba(X[feats])[:, 1]}))
    del X
sub = pd.concat(parts, ignore_index=True)
sub.to_csv(OUT_DIR / "submission.csv", index=False)
print("Da ghi", OUT_DIR / "submission.csv", sub.shape)
sub.describe()